In [1]:
import torch

from rlaopt.linalg import IdentityConfig, LinSys, NystromConfig
from rlaopt.solvers import PCG, PCGConfig, PCGStoppingCriteria

In [2]:
torch.set_default_dtype(torch.float64)

In [3]:
n = 10000
eigvals = torch.arange(1, n + 1) ** -2.0
reg = 1e-3

U = torch.randn(n, n)
U = torch.linalg.qr(U).Q

A = U @ torch.diag(eigvals) @ U.T
B = torch.randn(n, 10)

In [4]:
lin_sys = LinSys(A, B, reg)
lin_sys.cuda()

LinSys()

In [5]:
preconditioner_config_identity = IdentityConfig()
preconditioner_config_nystrom = NystromConfig(rank=100, base_damping=reg)

In [6]:
solver_config = PCGConfig(
    preconditioner_config=preconditioner_config_identity,
)

In [7]:
solver = PCG(lin_sys, solver_config)
params = lin_sys.w.clone()
state = solver.init_state(params)

In [8]:
max_iters = 100

for i in range(max_iters):
    params, state = solver.step(params, state)
    print(f"Iteration {i + 1}, residual norm: {state.res_norm}")

Iteration 1, residual norm: tensor([581.1992, 559.1131, 704.7878, 135.5411, 865.4481, 463.3894, 281.4192,
        236.0009, 442.0590, 394.5523], device='cuda:0',
       grad_fn=<LinalgVectorNormBackward0>)
Iteration 2, residual norm: tensor([20.6992, 20.5290, 21.9485, 15.6626, 23.5194,  9.6853, 14.6432, 18.3502,
        12.2942, 14.8205], device='cuda:0',
       grad_fn=<LinalgVectorNormBackward0>)
Iteration 3, residual norm: tensor([3.6543, 3.5312, 3.7921, 2.3162, 5.1462, 3.3781, 2.8236, 4.7065, 3.9883,
        2.8966], device='cuda:0', grad_fn=<LinalgVectorNormBackward0>)
Iteration 4, residual norm: tensor([0.7750, 0.5815, 0.9818, 0.5176, 1.2688, 0.6981, 0.6317, 1.3617, 1.0461,
        0.6786], device='cuda:0', grad_fn=<LinalgVectorNormBackward0>)
Iteration 5, residual norm: tensor([0.1888, 0.1186, 0.2060, 0.1122, 0.2182, 0.1611, 0.1254, 0.2984, 0.2008,
        0.1257], device='cuda:0', grad_fn=<LinalgVectorNormBackward0>)
Iteration 6, residual norm: tensor([0.1702, 0.3977, 0.1106, 0

In [9]:
params, final_res_norm = solver.solve(
    stopping_criteria=PCGStoppingCriteria(max_iters=100, tol=1e-10)
)
print(f"Final residual norm: {final_res_norm}")

Final residual norm: tensor([4.4767e-09, 9.6955e-09, 9.9508e-09, 1.4086e-09, 9.4163e-09, 2.6243e-09,
        3.4329e-09, 2.1600e-09, 4.7474e-09, 5.0528e-09], device='cuda:0',
       grad_fn=<LinalgVectorNormBackward0>)


In [10]:
lin_sys.compute_residual_norm(params, relative=True)

tensor([4.4778e-11, 9.6802e-11, 9.9022e-11, 1.3957e-11, 9.4958e-11, 2.6131e-11,
        3.4682e-11, 2.1655e-11, 4.7365e-11, 5.0219e-11], device='cuda:0',
       grad_fn=<DivBackward0>)